# Assignment 4: RAG System – AI/ML Knowledge Base Chatbot


---

## Overview
This notebook implements a complete Retrieval-Augmented Generation (RAG) system built on a curated collection of Wikipedia articles about Artificial Intelligence and Machine Learning topics.

### System Pipeline:
```
User Query → Embedding → Vector Search → Top-K Chunks → LLM (Gemini 1.5 Flash) → Answer
```

## Install Dependencies

In [1]:
import sys
!{sys.executable} -m pip install google-generativeai sentence-transformers faiss-cpu numpy scikit-learn rich --quiet

print(" Installation complete. Now restart your kernel using the ↻ button at the top!")

 Installation complete. Now restart your kernel using the ↻ button at the top!


---
## Part 1: Dataset Selection

### Dataset Description
The dataset consists of **5 Wikipedia articles** covering core topics in Artificial Intelligence and Machine Learning:

| File | Topic | Approx. Pages |
|------|-------|---------------|
| `machine_learning.txt` | Overview of ML, types, and history | ~3 pages |
| `deep_learning.txt` | Deep learning architectures and training | ~3 pages |
| `nlp.txt` | Natural Language Processing and RAG | ~3 pages |
| `neural_networks.txt` | Artificial neural networks | ~3 pages |
| `llm.txt` | Large Language Models | ~3 pages |

**Total: ~15 pages of curated technical content**

### Why This Dataset?
- **Relevance**: Directly related to the course material on AI/ML
- **Richness**: Dense with technical terms, concepts, and relationships — ideal for testing retrieval quality
- **Diversity**: Covers multiple interconnected topics enabling cross-document retrieval
- **Verifiability**: Wikipedia content is publicly verifiable, ensuring no privacy concerns

### Supported Question Types
- Factual: *"Who invented the LSTM network?"*
- Conceptual: *"What is the difference between supervised and unsupervised learning?"*
- Comparative: *"How does RAG reduce hallucination in LLMs?"*
- Applied: *"What activation function is most commonly used in hidden layers?"*
- Historical: *"When was the transformer architecture introduced?"*

## IMPORTS & API CONFIGURATION

In [2]:
# ─────────────────────────────────────────────────
# CELL 2: IMPORTS & API CONFIGURATION
# ─────────────────────────────────────────────────
import os
import re
import json
import numpy as np
import faiss
from pathlib import Path
from typing import List, Dict, Tuple
from sentence_transformers import SentenceTransformer
import google.generativeai as genai
from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich.markdown import Markdown
from rich import print as rprint

# 1. Setup API Key
# Replace "Your-Actual-Key-Here" 
os.environ["GOOGLE_API_KEY"] = "Your-Actual-Key-Here" 

# 2. Verify and Configure genai
if os.environ.get("GOOGLE_API_KEY"):
    genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
    console = Console()
    print(" API Key successfully configured")
    print(" All imports successful")
else:
    print(" Warning: No API Key found. Generation will fail.")

c:\Users\sheil\anaconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\sheil\AppData\Local\Temp\ipykernel_24392\4152618992.py:12: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


 API Key successfully configured
 All imports successful


---
## Part 2: Preprocessing – Loading, Cleaning, and Chunking

In [3]:
# ─────────────────────────────────────────────────
# STEP 2.1: Load raw documents from the dataset folder
# ─────────────────────────────────────────────────

DATASET_DIR = Path("dataset")

def load_documents(dataset_dir: Path) -> List[Dict]:
    """Load all .txt files from the dataset directory."""
    documents = []
    for filepath in sorted(dataset_dir.glob("*.txt")):
        with open(filepath, "r", encoding="utf-8") as f:
            raw_text = f.read()
        documents.append({
            "filename": filepath.name,
            "topic": filepath.stem.replace("_", " ").title(),
            "raw_text": raw_text,
            "char_count": len(raw_text)
        })
        print(f"   Loaded: {filepath.name} ({len(raw_text):,} characters)")
    return documents

documents = load_documents(DATASET_DIR)
print(f"\n Loaded {len(documents)} documents, total characters: {sum(d['char_count'] for d in documents):,}")

   Loaded: deep_learning.txt (7,041 characters)
   Loaded: llm.txt (6,997 characters)
   Loaded: machine_learning.txt (7,176 characters)
   Loaded: neural_networks.txt (6,708 characters)
   Loaded: nlp.txt (7,413 characters)

 Loaded 5 documents, total characters: 35,335


In [4]:
import os
print(f"Current Directory: {os.getcwd()}")
print(f"Files in this folder: {os.listdir('.')}")

Current Directory: c:\Centennial\WinterSemester2026\DeepLearning\RAG-Pipeline
Files in this folder: ['app.py', 'chunks_metadata.json', 'dataset', 'faiss_index.bin', 'rag_system.ipynb', 'README (1).md', 'requirements.txt']


In [5]:
# ─────────────────────────────────────────────────
# STEP 2.2: Clean text
# ─────────────────────────────────────────────────

def clean_text(text: str) -> str:
    """
    Clean raw text by:
    - Removing source/URL header lines
    - Collapsing multiple blank lines into single line breaks
    - Stripping leading/trailing whitespace from lines
    - Removing non-ASCII characters
    """
    # Remove source/URL header lines (first 2 lines)
    lines = text.split("\n")
    # Keep lines after the header (skip "Source:" and "URL:" lines)
    cleaned_lines = []
    skip_header = True
    for line in lines:
        if skip_header and (line.startswith("Source:") or line.startswith("URL:") or line.strip() == ""):
            continue
        skip_header = False
        cleaned_lines.append(line)
    
    text = "\n".join(cleaned_lines)
    
    # Remove non-ASCII characters
    text = text.encode("ascii", errors="ignore").decode("ascii")
    
    # Collapse multiple blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)
    
    # Strip excessive whitespace within lines
    text = re.sub(r"[ \t]{2,}", " ", text)
    
    return text.strip()

for doc in documents:
    doc["clean_text"] = clean_text(doc["raw_text"])
    reduction = (1 - len(doc["clean_text"]) / doc["char_count"]) * 100
    print(f"   {doc['filename']}: {doc['char_count']:,} → {len(doc['clean_text']):,} chars ({reduction:.1f}% reduced)")

print("\n Text cleaning complete")

   deep_learning.txt: 7,041 → 6,955 chars (1.2% reduced)
   llm.txt: 6,997 → 6,896 chars (1.4% reduced)
   machine_learning.txt: 7,176 → 7,085 chars (1.3% reduced)
   neural_networks.txt: 6,708 → 6,568 chars (2.1% reduced)
   nlp.txt: 7,413 → 7,299 chars (1.5% reduced)

 Text cleaning complete


In [6]:
# ─────────────────────────────────────────────────
# STEP 2.3: Chunking
# ─────────────────────────────────────────────────

"""
CHUNKING STRATEGY JUSTIFICATION
================================
Chunk Size: 500 characters
  - Small enough to be semantically focused on one concept
  - Large enough to provide sufficient context for answering questions
  - Fits well within embedding model max sequence lengths (~512 tokens)

Overlap: 100 characters (20% of chunk size)
  - Prevents concepts split at chunk boundaries from being missed
  - Ensures continuity so retrieving one chunk still captures context from adjacent text
  - 20% overlap is a widely recommended balance between context continuity and redundancy
"""

CHUNK_SIZE = 500      # characters per chunk
CHUNK_OVERLAP = 100   # character overlap between chunks

def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> List[str]:
    """
    Split text into overlapping chunks.
    Tries to break at paragraph or sentence boundaries when possible.
    """
    chunks = []
    start = 0
    text_length = len(text)
    
    while start < text_length:
        end = start + chunk_size
        
        if end >= text_length:
            # Last chunk
            chunks.append(text[start:].strip())
            break
        
        # Try to break at paragraph boundary
        para_break = text.rfind("\n\n", start, end)
        if para_break != -1 and para_break > start + chunk_size // 2:
            end = para_break
        else:
            # Try to break at sentence boundary (period + space)
            sentence_break = text.rfind(". ", start, end)
            if sentence_break != -1 and sentence_break > start + chunk_size // 2:
                end = sentence_break + 1
        
        chunk = text[start:end].strip()
        if chunk:  # skip empty chunks
            chunks.append(chunk)
        
        # Move start forward with overlap
        start = end - overlap
    
    return chunks


# Build the full chunk corpus
all_chunks = []
chunk_metadata = []

for doc in documents:
    chunks = chunk_text(doc["raw_text"])
    doc["chunks"] = chunks
    doc["num_chunks"] = len(chunks)
    
    for i, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        chunk_metadata.append({
            "source": doc["filename"],
            "topic": doc["topic"],
            "chunk_index": i,
            "total_chunks": len(chunks)
        })
    
    print(f"   {doc['filename']}: {len(chunks)} chunks")

print(f"\n Total chunks created: {len(all_chunks)}")
print(f"   Average chunk size: {np.mean([len(c) for c in all_chunks]):.0f} characters")
print(f"   Min / Max: {min(len(c) for c in all_chunks)} / {max(len(c) for c in all_chunks)} characters")

   deep_learning.txt: 24 chunks
   llm.txt: 24 chunks
   machine_learning.txt: 25 chunks
   neural_networks.txt: 24 chunks
   nlp.txt: 25 chunks

 Total chunks created: 122
   Average chunk size: 385 characters
   Min / Max: 120 / 500 characters


In [7]:
# Display a sample chunk to verify quality
print(" Sample chunk preview (Chunk #5):")
print("-" * 60)
print(all_chunks[5])
print("-" * 60)
print(f"Source: {chunk_metadata[5]['source']} | Index: {chunk_metadata[5]['chunk_index']}")

 Sample chunk preview (Chunk #5):
------------------------------------------------------------
s problem affected training of deep networks, as gradients became very small during backpropagation.

In 1997, Hochreiter and Schmidhuber proposed long short-term memory (LSTM), which overcame the vanishing gradient problem for recurrent networks. LSTM became widely used for sequence-to-sequence tasks.
------------------------------------------------------------
Source: deep_learning.txt | Index: 5


---
## Part 3: Embeddings – Generate and Store in FAISS Vector Database

In [8]:
# ─────────────────────────────────────────────────
# STEP 3.1: Load Embedding Model
# ─────────────────────────────────────────────────

# Using 'all-MiniLM-L6-v2' – a lightweight but powerful sentence transformer
# Produces 384-dimensional embeddings, fast, and well-suited for semantic similarity

print(" Loading embedding model: all-MiniLM-L6-v2 ...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
EMBEDDING_DIM = 384  # Output dimension for this model
print(f" Embedding model loaded. Dimension: {EMBEDDING_DIM}")

 Loading embedding model: all-MiniLM-L6-v2 ...


C:\Users\sheil\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


 Embedding model loaded. Dimension: 384


In [9]:
# ─────────────────────────────────────────────────
# STEP 3.2: Generate Embeddings for All Chunks
# ─────────────────────────────────────────────────

print(f" Generating embeddings for {len(all_chunks)} chunks...")

# Encode in batches for efficiency
chunk_embeddings = embedding_model.encode(
    all_chunks,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True   # L2 normalize for cosine similarity via dot product
)

print(f"\n Embeddings shape: {chunk_embeddings.shape}")
print(f"   Each chunk → {chunk_embeddings.shape[1]}-dim vector")

 Generating embeddings for 122 chunks...


Batches: 100%|██████████| 4/4 [00:06<00:00,  1.64s/it]


 Embeddings shape: (122, 384)
   Each chunk → 384-dim vector


In [10]:
# ─────────────────────────────────────────────────
# STEP 3.3: Build FAISS Vector Index
# ─────────────────────────────────────────────────

# Using IndexFlatIP (Inner Product) on normalized vectors = cosine similarity
# Good choice for our corpus size (~100 chunks). For larger corpora, use IndexIVFFlat.

index = faiss.IndexFlatIP(EMBEDDING_DIM)
index.add(chunk_embeddings.astype(np.float32))

print(f" FAISS index built")
print(f"   Index type: IndexFlatIP (exact cosine similarity)")
print(f"   Vectors stored: {index.ntotal}")
print(f"   Embedding dimension: {EMBEDDING_DIM}")

# Save the index for later use (e.g., Streamlit app)
faiss.write_index(index, "faiss_index.bin")
with open("chunks_metadata.json", "w") as f:
    json.dump({"chunks": all_chunks, "metadata": chunk_metadata}, f, indent=2)

print("\n Index saved to: faiss_index.bin")
print(" Metadata saved to: chunks_metadata.json")

 FAISS index built
   Index type: IndexFlatIP (exact cosine similarity)
   Vectors stored: 122
   Embedding dimension: 384

 Index saved to: faiss_index.bin
 Metadata saved to: chunks_metadata.json


---
## Part 4: Retrieval – Query the Vector Database

In [11]:
# ─────────────────────────────────────────────────
# STEP 4.1: Define Retrieval Function (Updated)
# ─────────────────────────────────────────────────

TOP_K = 4

def retrieve_chunks(query: str, top_k: int = TOP_K) -> List[Dict]:
    """
    Uses the global index, embedding_model, and all_chunks 
    to find the most relevant information.
    """
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    scores, indices = index.search(query_embedding.astype(np.float32), top_k)
    results = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0])):
        if idx == -1:
            continue
        results.append({
            "rank": rank + 1,
            "score": float(score),
            "chunk_text": all_chunks[idx],
            "source": chunk_metadata[idx]["source"],
            "topic": chunk_metadata[idx]["topic"],
            "chunk_index": chunk_metadata[idx]["chunk_index"]
        })
    return results


def display_retrieved_chunks(query: str, results: List[Dict]):
    console.print(f"\n[bold cyan]Query:[/bold cyan] {query}")
    console.print(f"[bold]Retrieved {len(results)} chunks:[/bold]\n")
    for r in results:
        console.print(Panel(
            r["chunk_text"],
            title=f"[green]Rank {r['rank']} | Score: {r['score']:.4f} | Source: {r['source']}[/green]",
            expand=False
        ))

print("Retrieval function updated and standardized")
print("display_retrieved_chunks function defined")

Retrieval function updated and standardized
display_retrieved_chunks function defined


In [12]:
# ─────────────────────────────────────────────────
# STEP 4.2: Test Retrieval with a Sample Query
# ─────────────────────────────────────────────────

test_query = "What is the difference between supervised and unsupervised learning?"
results = retrieve_chunks(test_query)
display_retrieved_chunks(test_query, results)

Query: What is the difference between supervised and unsupervised learning?

Retrieved 4 chunks:

╭───────────────────────────── Rank 1 | Score: 0.6107 | Source: machine_learning.txt ─────────────────────────────╮
│ decision trees, random forests, support vector machines, and neural networks.                                   │
│                                                                                                                 │
│ Unsupervised learning                                                                                           │
│                                                                                                                 │
│ Unsupervised learning algorithms find structures in data that has not been labeled, classified or categorized.  │
│ Instead of responding to feedback, unsupervised learning algorithms identify commonalities in the data and      │
│ react based on the presence or absence of such commonalities in each new piece of data.                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────── Rank 2 | Score: 0.5789 | Source: machine_learning.txt ─────────────────────────────╮
│ the data and react based on the presence or absence of such commonalities in each new piece of data.            │
│                                                                                                                 │
│ Common examples of unsupervised learning algorithms include k-means clustering, DBSCAN, hierarchical            │
│ clustering, and autoencoders. Principal Component Analysis (PCA) is also an unsupervised technique used for     │
│ dimensionality reduction.                                                                                       │
│                                                                                                                 │
│ Reinforcement learning                                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────── Rank 3 | Score: 0.5170 | Source: machine_learning.txt ─────────────────────────────╮
│ ated world champion Go players, and OpenAI's work on robotic manipulation.                                      │
│                                                                                                                 │
│ Semi-supervised learning                                                                                        │
│                                                                                                                 │
│ Semi-supervised learning falls between unsupervised learning (without any labeled training data) and supervised │
│ learning (with completely labeled training data). It makes use of a small amount of labeled data and a large    │
│ amount of unlabeled data during training. This approach is useful when labeling data is expensive or            │
│ time-consuming.                                                                                                 │
│                                                                                                                 │
│ Self-supervised learning                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────── Rank 4 | Score: 0.4803 | Source: machine_learning.txt ─────────────────────────────╮
│ gin, marking the beginning of the deep learning era.                                                            │
│                                                                                                                 │
│ Types of Machine Learning                                                                                       │
│                                                                                                                 │
│ Supervised learning                                                                                             │
│                                                                                                                 │
│ Supervised learning algorithms build a mathematical model of a set of data that contains both the inputs and    │
│ the desired outputs. The data, known as training data, consists of a set of training examples. Each training    │
│ example has one or more inputs and the desired output, also known as a supervisory signal.                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
## Part 5: Generation – LLM-Based Answer Generation Using Google Gemini

In [13]:
# ─────────────────────────────────────────────────
# STEP 5.1: Configure the Google Gemini Client
# ─────────────────────────────────────────────────
import google.generativeai as genai
import os

# 1. Try to get the key from environment variables
api_key = os.environ.get("GOOGLE_API_KEY")

# 2. Fallback: If environment variable is empty, set it manually here
if not api_key or api_key == "":
    # Replace the text below with your actual AIza... key during testing
    api_key = "Your-Actual-API-Key" 
    os.environ["GOOGLE_API_KEY"] = api_key

# 3. Configure the library
if api_key:
    genai.configure(api_key=api_key)
    LLM_MODEL = "gemini-1.5-flash"
    print(f" Gemini client configured | Model: {LLM_MODEL}")
else:
    print(" ERROR: No API Key found. Please set GOOGLE_API_KEY.")

 Gemini client configured | Model: gemini-1.5-flash


In [14]:
# ─────────────────────────────────────────────────
# STEP 5.2: CONSOLIDATED RAG PIPELINE
# ─────────────────────────────────────────────────


import google.generativeai as genai

def generate_answer(query: str, retrieved_chunks: List[Dict]) -> str:
    # 1. Use the 2026 model aliases (Stable & Production-ready)
    # Primary: gemini-2.0-flash | Backup: gemini-flash-latest
    MODELS_TO_TRY = ['gemini-2.0-flash', 'gemini-2.5-flash', 'gemini-flash-latest']
    
    context_text = "\n\n".join([f"Source {c['source']}: {c['chunk_text']}" for c in retrieved_chunks])
    
    prompt = f"""You are a technical AI assistant. Use the context to answer the question.
    Cite your sources (e.g., [nlp.txt]). If unknown, say so. Answer ONLY using information from the provided context documents.
    Do NOT use any external knowledge or make up information.
    If the context lacks information, say so explicitly.
    Cite the source document in your answer.

    CONTEXT:
    {context_text}

    QUESTION: {query}
    
    ANSWER:"""

    for model_name in MODELS_TO_TRY:
        try:
            model = genai.GenerativeModel(model_name)
            response = model.generate_content(prompt)
            return response.text
        except Exception:
            continue # Try the next model if one fails
            
    return " Error: All Gemini models (2.0/2.5) returned 404. Please check your API key permissions."


def rag_query(query: str, top_k: int = 3, verbose: bool = False):
    """The full Retrieve-Augment-Generate flow."""
    # 1. Retrieve
    retrieved = retrieve_chunks(query=query, top_k=top_k)
    
    if verbose:
        rprint(f"[italic gray]Found {len(retrieved)} relevant segments...[/italic gray]")
    
    # 2. Generate
    answer = generate_answer(query, retrieved)
    
    # Returning "question" key to satisfy Step 6.2 loop
    return {
        "question": query,
        "query": query,
        "retrieved": retrieved,
        "answer": answer
    }

print(" RAG Pipeline (Step 5.2) synchronized with Evaluation requirements.")

 RAG Pipeline (Step 5.2) synchronized with Evaluation requirements.


## Initialize the Embedding Model

In [15]:
from sentence_transformers import SentenceTransformer

# This name MUST match the one you used to create your FAISS index
model_name = "all-MiniLM-L6-v2"

print(f" Loading embedding model: {model_name}...")
embed_model = SentenceTransformer(model_name)
print(" embed_model is now defined and ready!")

 Loading embedding model: all-MiniLM-L6-v2...


C:\Users\sheil\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


 embed_model is now defined and ready!


In [16]:
# Test the full RAG pipeline
result = rag_query("What is backpropagation and how does it work?")

---
## Part 6: Evaluation – Testing with 7 Questions

In [17]:
# ─────────────────────────────────────────────────
# STEP 6.1: Define evaluation questions
# ─────────────────────────────────────────────────

# 7 evaluation questions spanning different topics, difficulty levels, and question types
EVAL_QUESTIONS = [
    {
        "id": 1,
        "question": "Who invented the LSTM network and what problem does it solve?",
        "expected_topics": ["LSTM", "Hochreiter", "vanishing gradient"],
        "type": "Factual"
    },
    {
        "id": 2,
        "question": "What is the difference between supervised and unsupervised learning?",
        "expected_topics": ["labeled data", "supervised", "unsupervised", "clustering"],
        "type": "Conceptual"
    },
    {
        "id": 3,
        "question": "How does Retrieval-Augmented Generation (RAG) reduce hallucination in LLMs?",
        "expected_topics": ["retrieval", "context", "hallucination", "grounding"],
        "type": "Applied"
    },
    {
        "id": 4,
        "question": "What activation function is most commonly used in hidden layers of deep networks and why?",
        "expected_topics": ["ReLU", "vanishing gradient", "activation"],
        "type": "Conceptual"
    },
    {
        "id": 5,
        "question": "When was the transformer architecture introduced and what is its key innovation?",
        "expected_topics": ["2017", "attention", "Vaswani", "self-attention"],
        "type": "Factual"
    },
    {
        "id": 6,
        "question": "What are the main types of neural network architectures and their use cases?",
        "expected_topics": ["CNN", "RNN", "Transformer", "image", "sequence"],
        "type": "Comparative"
    },
    {
        "id": 7,
        "question": "What is overfitting and what techniques are used to prevent it?",
        "expected_topics": ["overfitting", "dropout", "regularization", "generalization"],
        "type": "Applied"
    }
]

print(f" {len(EVAL_QUESTIONS)} evaluation questions defined")

 7 evaluation questions defined


In [18]:
# ─────────────────────────────────────────────────
# STEP 6.2: Run Evaluation (WITH VISUAL OUTPUT)
# ─────────────────────────────────────────────────

evaluation_results = []

for q in EVAL_QUESTIONS:
    console.rule(f"[bold]Question {q['id']} of {len(EVAL_QUESTIONS)} | Type: {q['type']}[/bold]")
    
    # 1. Run the RAG pipeline
    result = rag_query(q["question"], verbose=True)
    
    # 2. Store metadata
    result["type"] = q["type"]
    result["expected_topics"] = q["expected_topics"]
    evaluation_results.append(result)
    
    # 3. ADD THIS: Display the full answer in a panel
    from rich.panel import Panel
    console.print(Panel(result["answer"], title=f"[bold green]Answer[/bold green]", expand=False))
    console.print("\n") # Add a little breathing room

───────────────────────────────────────── Question 1 of 7 | Type: Factual ─────────────────────────────────────────

Found 3 relevant segments...

╭───────────────────────────────────────── Answer ─────────────────────────────────────────╮
│  Error: All Gemini models (2.0/2.5) returned 404. Please check your API key permissions. │
╰──────────────────────────────────────────────────────────────────────────────────────────╯

─────────────────────────────────────── Question 2 of 7 | Type: Conceptual ────────────────────────────────────────

Found 3 relevant segments...

...

Found 3 relevant segments...

╭───────────────────────────────────────── Answer ─────────────────────────────────────────╮
│  Error: All Gemini models (2.0/2.5) returned 404. Please check your API key permissions. │
╰──────────────────────────────────────────────────────────────────────────────────────────╯

───────────────────────────────────────── Question 7 of 7 | Type: Applied ─────────────────────────────────────────

Found 3 relevant segments...

╭───────────────────────────────────────── Answer ─────────────────────────────────────────╮
│  Error: All Gemini models (2.0/2.5) returned 404. Please check your API key permissions. │
╰──────────────────────────────────────────────────────────────────────────────────────────╯

In [19]:
# ─────────────────────────────────────────────────
# STEP 6.3: Accuracy Assessment
# ─────────────────────────────────────────────────

def assess_accuracy(result: Dict) -> str:
    """Keyword-based accuracy estimation."""
    answer_lower = result.get("answer", "").lower()
    matched = [kw for kw in result.get("expected_topics", []) if kw.lower() in answer_lower]
    if not result.get("expected_topics"):
        return "N/A"
    match_ratio = len(matched) / len(result["expected_topics"])
    if match_ratio >= 0.7:
        return "Correct"
    elif match_ratio >= 0.4:
        return "Partial"
    else:
        return "Incorrect"

# Build evaluation summary table
table = Table(title="RAG System Evaluation Results", show_lines=True)
table.add_column("#", style="cyan", width=4)
table.add_column("Question", style="white", width=45)
table.add_column("Type", style="yellow", width=12)
table.add_column("Top Source", style="green", width=20)
table.add_column("Accuracy", style="green", width=12)

for i, result in enumerate(evaluation_results):
    accuracy = assess_accuracy(result)
    chunks = result.get("retrieved", [])
    top_source = chunks[0]["source"] if chunks else "N/A"
    table.add_row(
        str(i + 1),
        result.get("query", "Unknown")[:80],
        result.get("type", "N/A"),
        top_source,
        accuracy
    )

console.print(table)

# Summary stats
accuracies = [assess_accuracy(r) for r in evaluation_results]
correct   = sum(1 for a in accuracies if "Correct" in a)
partial   = sum(1 for a in accuracies if "Partial" in a)
incorrect = sum(1 for a in accuracies if "Incorrect" in a)

console.print(f"\n[bold]Summary:[/bold] {correct} Correct | {partial} Partial | {incorrect} Incorrect")
console.print(f"[bold green]Overall Accuracy: {(correct + 0.5*partial)/len(evaluation_results)*100:.0f}%[/bold green]")

                                        RAG System Evaluation Results                                        
┏━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ #    ┃ Question                                      ┃ Type         ┃ Top Source           ┃ Accuracy     ┃
┡━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ 1    │ Who invented the LSTM network and what        │ Factual      │ deep_learning.txt    │ Incorrect    │
│      │ problem does it solve?                        │              │                      │              │
├──────┼───────────────────────────────────────────────┼──────────────┼──────────────────────┼──────────────┤
│ 2    │ What is the difference between supervised and │ Conceptual   │ machine_learning.txt │ Incorrect    │
│      │ unsupervised learning?                        │              │                      │              │
├──────┼───────────────────────────────────────────────┼──────────────┼──────────────────────┼──────────────┤
│ 3    │ How does Retrieval-Augmented Generation (RAG) │ Applied      │ llm.txt              │ Incorrect    │
│      │ reduce hallucination in LLMs?                 │              │                      │              │
├──────┼───────────────────────────────────────────────┼──────────────┼──────────────────────┼──────────────┤
│ 4    │ What activation function is most commonly     │ Conceptual   │ deep_learning.txt    │ Incorrect    │
│      │ used in hidden layers of deep networks        │              │                      │              │
├──────┼───────────────────────────────────────────────┼──────────────┼──────────────────────┼──────────────┤
│ 5    │ When was the transformer architecture         │ Factual      │ deep_learning.txt    │ Incorrect    │
│      │ introduced and what is its key innovation?    │              │                      │              │
├──────┼───────────────────────────────────────────────┼──────────────┼──────────────────────┼──────────────┤
│ 6    │ What are the main types of neural network     │ Comparative  │ deep_learning.txt    │ Incorrect    │
│      │ architectures and their use cases?            │              │                      │              │
├──────┼───────────────────────────────────────────────┼──────────────┼──────────────────────┼──────────────┤
│ 7    │ What is overfitting and what techniques are   │ Applied      │ machine_learning.txt │ Incorrect    │
│      │ used to prevent it?                           │              │                      │              │
└──────┴───────────────────────────────────────────────┴──────────────┴──────────────────────┴──────────────┘

Summary: 0 Correct | 0 Partial | 7 Incorrect

Overall Accuracy: 0%

In [20]:
res = evaluation_results[0]
print(f"Keywords we wanted: {res['expected_topics']}")
print(f"What Gemini said: {res['answer'][:200]}...")

# Test the match manually
for kw in res['expected_topics']:
    print(f"Is '{kw}' in the answer? {'Yes' if kw.lower() in res['answer'].lower() else 'No'}")

Keywords we wanted: ['LSTM', 'Hochreiter', 'vanishing gradient']
What Gemini said:  Error: All Gemini models (2.0/2.5) returned 404. Please check your API key permissions....
Is 'LSTM' in the answer? No
Is 'Hochreiter' in the answer? No
Is 'vanishing gradient' in the answer? No
